In [1]:
using Pkg
Pkg.activate(".")
using Meshes
import WGLMakie as Mke
using GeoIO
using GeoStats
using LinearAlgebra
using DataFrames

  Activating project at `~/Documents/GitHub/Julia_Tutorials/HeatMethod`


In [130]:
bunny = GeoIO.load("./bunny.obj").geometry;
bunny = Rotate(Vec(0, 1, 0), Vec(0, 0, 1))(bunny)
bunny = Rotate(Vec(1, 0, 0), Vec(-1, 1, 0))(bunny);

bunny = bunny |> Scale(100)

bunny = topoconvert(HalfEdgeTopology, bunny)
viz(bunny, showsegments=true)

In [90]:
l = laplacematrix(bunny, kind=:cotangent);

In [91]:
function edges(t::Triangle)
    v = vertices(t)
    return [v[3] - v[2],v[1] - v[3], v[2] - v[1]]
end
e1, e2, e3 = edges(bunny[1])

3-element Vector{Vec{3, Quantity{Float64, 𝐋, Unitful.FreeUnits{(m,), 𝐋, nothing}}}}:
 Vec(-0.49390630848421146 m, -0.07967184235663183 m, -0.14339299999999966 m)
 Vec(0.5995226057493586 m, 0.15596159297884782 m, -0.2779309999999988 m)
 Vec(-0.1056162972651471 m, -0.07628975062221599 m, 0.4213239999999985 m)

In [125]:
function mean_edges(mesh)
    map(collect(mesh.topology.edge4pair)) do (v,e)
        norm(mesh.vertices[v[2]] - mesh.vertices[v[1]])
    end |> mean
end

mean_edges(bunny)

0.5592992006318498 m

In [126]:
function normalvector(t::Triangle)
    v = vertices(t)
    n = ((v[2] - v[1]) × (v[3] - v[1]))/ norm((v[2] - v[1]) × (v[3] - v[1]))
    return n
end

normalvector(bunny[1])

function normalsegment(t::Triangle)
    n = normalvector(t)
    p = centroid(t)
    np = Vec(n) +  p - Point(0,0,0)
    np = Point(np.coords)
    segment = Segment(p,np)
    return segment
end

ns = normalsegment.(bunny);
v = viz(bunny)
viz!(ns,color=:red)

v

In [129]:
# to(Point(0,1,1))
typeof(bunny)

SimpleMesh{𝔼{3}, Cartesian3D{NoDatum, Quantity{Float64, 𝐋, Unitful.FreeUnits{(m,), 𝐋, nothing}}}, Vector{Point{𝔼{3}, Cartesian3D{NoDatum, Quantity{Float64, 𝐋, Unitful.FreeUnits{(m,), 𝐋, nothing}}}}}, HalfEdgeTopology}